# 02 — Evaluate the trained model

**Purpose:** load the checkpoint from `01_train.ipynb`, compute held-out test metrics on chromosomes 1 and 2, and produce the numbers and plots for the README.

This notebook is a **thin consumer** of `src/evaluate.py`, all metric logic lives in `src/`. The notebook's job is to make the results legible.

**Metric definitions (for the README):**
- **AUROC** — area under the ROC curve. Threshold-independent ranking quality. 0.5 = chance, 1.0 = perfect.
- **AUPRC** — area under the precision-recall curve. With balanced classes (1:1), it should track AUROC closely.
- **Accuracy / precision / recall** — at a fixed decision threshold (default 0.5).

**Held-out set:** chromosomes `1` and `2`, never seen during training.

**Expected results:** a 1D CNN on CREB1 ChIP-seq typically achieves AUROC **0.85–0.95** on held-out chromosomes. AUROC near 0.5 means the model didn't learn, go back to `00_data_sanity.ipynb` before touching hyperparameters. AUROC > 0.98 is suspicious, check for train/test leakage.

**Note on the training run this evaluates:** the 20-epoch run showed val loss minimum at epoch 7, with overfitting after. The checkpoint saved is the **epoch-7 best-val weights**, not the epoch-20 weights. This is intentional and correct.

In [1]:
import sys, os
from pathlib import Path

# Make `src` importable regardless of Jupyter's CWD
p = Path.cwd()
while p != p.parent and not (p / "src").exists():
    p = p.parent
os.chdir(p)
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
print("repo root:", p)

%load_ext autoreload
%autoreload 2

import json
import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset

from src.data import DataConfig, build_dataset, split_by_chromosome
from src.model import Creb1CNN
from src.train import load_checkpoint
from src.evaluate import predict, compute_metrics, curves, evaluate_by_chromosome

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

repo root: /home/lux/TFBS-creb1-predicter
torch: 2.15.0+cu132
cuda available: True


## 1. Configuration and checkpoint

Same `DataConfig` and `test_chroms` as `01_train.ipynb`, **consistency here is critical.** If you change `test_chroms` between training and evaluation, you're evaluating on data the model trained on.

`load_checkpoint` returns a dict with `state_dict` plus metadata (`epoch`, `val_loss`) saved during training. The `epoch` tells you which training epoch produced the best-val weights.

In [2]:
cfg = DataConfig(window=500, neg_per_pos=1, seed=0, test_chroms=("1", "2"))
CKPT = "checkpoints/cnn.pt"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
THRESHOLD = 0.5

model = Creb1CNN()
meta = load_checkpoint(CKPT, model)
print(f"checkpoint: {CKPT}")
print(f"  best epoch:    {meta.get('epoch')}")
print(f"  best val loss: {meta.get('val_loss'):.4f}")
print(f"  device:        {DEVICE}")

checkpoint: checkpoints/cnn.pt
  best epoch:    7
  best val loss: 0.1247
  device:        cuda


## 2. Load the held-out test set

`split_by_chromosome` with `test_chroms=("1", "2")` produces a test set from **only those chromosomes**. Every row in `X_te` comes from a chromosome the model never saw during training — this is the leak-free evaluation that makes the AUROC meaningful.

**What to check:** `positives` and `negatives` counts should be roughly balanced (the test set carries the 1:1 pos/neg ratio from training data).